In [ ]:
# We should import langchain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [ ]:
# now to load the files from the directory
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4.1-nano"
db_name = "vector_db"

openai_api_key = os.getenv("OPENAI_API_KEY")


In [ ]:
model = "gpt-4.1-nano"
db_name = "vector_db_1"

knowledge_base_path = "**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Total files found: {len(files)}")

In [ ]:
content = ""
for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        content+=f.read()
        content+=""  # Adding some space between files

print(f"Total characters in all files: {len(content)}")

        

In [ ]:
encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(content)
print(f"Total tokens in all files: {len(tokens)}")

In [ ]:
import pydantic
from pydantic import BaseModel
class Result(BaseModel):
    metadata: dict
    page_content: str

In [ ]:
filepaths = glob.glob("**/*.md", recursive=True)

documents = []
for filepath in filepaths:
    doc_type = filepath.split('/')[0]
    with open(filepath,'r', encoding='utf-8') as f:
        text = f.read()
        metadata = {"doc_type": doc_type}
        documents.append(Result(metadata=metadata, page_content=text))
print(f"Total documents loaded: {len(documents)}")
    

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
len(chunks)

In [ ]:
# Now we will check out agentic AI chunking

def make_chunks(document, target_size=500, overlap=200):
    meta = getattr(document, "metadata", {}) or {}
    doc_type = meta.get("doc_type", "unknown")
    content = getattr(document, "page_content", "") or ""

    prompt = (
        "You are an expert agent for splitting documents into coherent chunks for embeddings and retrieval."
        f"Document metadata: {meta}"
        "Requirements:"
        f"- Target chunk size: approx {target_size} characters with ~{overlap} characters overlap and Totally we have 76 documents which will be passed one by one."
        "- Prefer splitting at headings, section breaks, blank lines, or list boundaries."
        "- Do not split inside sentences; keep chunks semantically coherent."
        "- Preserve heading text at the start of a chunk when possible."
        "- For each chunk, produce a JSON object with fields:"
        "  {\"id\": \"<doc_type>_<index>\", \"doc_type\": <doc_type>,"
        "\"chunk_text\": <string>, \"summary\": <one-sentence string>, \"keywords\": [<3 terms>]"
        "- Return a JSON (only) of these chunk objects and make sure to place the commas at end of each json object"
        "Now split the following document content and produce the JSON output."
        "-----DOCUMENT START-----"
        f"{content}"
        "-----DOCUMENT END-----"
    )

    llm = ChatOpenAI(temperature=0,model_name=MODEL)
    response = llm.invoke(prompt)

    return response.content

In [ ]:
make_chunks(documents[0])

In [ ]:
arr = [{   "id": "products_1",    "doc_type": "products",    "start_char": 0,    "end_char": 453,    "chunk_text": "# Product Summary\\# Rellm: AI-Powered Enterprise Reinsurance Solution\\## Summary\\Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.",    "summary": "Introduction to Rellm as an AI-driven reinsurance platform by Insurellm highlighting its purpose and benefits.",    "keywords": ["Reinsurance", "AI", "Risk Management", "Analytics", "Insurellm"],    "approx_tokens": 180  },  {    "id": "products_2",    "doc_type": "products",    "start_char": 453,    "end_char": 1024,    "chunk_text": "## Features\\### AI-Driven Analytics\Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\\### Seamless Integrations\Rellm\'s architecture is designed for effortless integration with existing systems. Whether it\'s policy management, claims processing, or financial reporting, Rellm connects seamlessly with diverse data sources to create a unified ecosystem.\\### Risk Assessment Module\The comprehensive risk assessment module within Rellm allows insurers to evaluate risk profiles accurately. By leveraging historical data and advanced modeling techniques, Rellm provides a clear picture of potential liabilities and expected outcomes.\\### Customizable Dashboard\Rellm features a customizable dashboard that presents key metrics and performance indicators in an intuitive interface. Users can tailor their view to focus on what matters most to their business, enhancing user experience and productivity.\\### Regulatory Compliance Tools\Rellm includes built-in compliance tracking features to help organizations meet local and international regulatory standards. This ensures that reinsurance practices remain transparent and accountable.\\### Client and Broker Portals\Rellm offers dedicated portals for both clients and brokers, facilitating real-time communication and documentation sharing. This strengthens partnerships and drives operational excellence across the board.",    "summary": "Detailed features of Rellm, including analytics, integrations, risk assessment, dashboards, compliance, and portals.",    "keywords": ["Features", "Analytics", "Integrations", "Risk Assessment", "Dashboards", "Compliance", "Portals"],    "approx_tokens": 330  },  {    "id": "products_3",    "doc_type": "products",    "start_char": 1024,    "end_char": 1574,    "chunk_text": "## Pricing\\Insurellm offers flexible pricing plans for Rellm to cater to various business needs:\\- **Basic Plan**: $5,000/month\  - Includes access to core features and standard integrations.\  \- **Professional Plan**: $10,000/month\  - Includes all features, advanced integrations, and priority customer support.\  \- **Enterprise Plan**: Custom pricing\  - Tailored solutions with personalized features, extensive integrations, and dedicated account management.\\Join the growing number of organizations leveraging Rellm to enhance their reinsurance processes while driving profitability and compliance.",    "summary": "Overview of Rellm\'s flexible pricing plans: Basic, Professional, and Enterprise options.",    "keywords": ["Pricing", "Plans", "Basic", "Professional", "Enterprise"],    "approx_tokens": 150  },  {    "id": "products_4",    "doc_type": "products",    "start_char": 1574,    "end_char": 2094,    "chunk_text": "## 2025-2026 Roadmap\\At Insurellm, we are committed to the continuous improvement of Rellm. Our roadmap for 2025-2026 includes:\\- **Q3 2025**: \  - Launch of the Rellm Mobile App for on-the-go insights and management.\  - Introduction of augmented reality (AR) features for interactive risk assessments.\\- **Q1 2026**: \  - Deployment of advanced machine learning models for even more accurate risk predictions.\  - Expansion of integration capabilities to support emerging technologies in the insurance sector.\\- **Q3 2026**: \  - Release of a community platform for Rellm users to exchange insights, tips, and best practices.\  - Launch of Rellm 2.0, featuring enhanced user interface and premium features based on user feedback.\\Experience the future of reinsurance with Rellm, where innovation meets reliability. Let Insurellm help you navigate the complexities of the reinsurance market smarter and faster.",    "summary": "Roadmap for Rellm\'s development from 2025 to 2026, including app launches, AR features, ML models, community platform, and Rellm 2.0.",    "keywords": ["Roadmap", "2025", "2026", "Mobile App", "AR", "Machine Learning", "Community Platform"]}]

In [ ]:
from tqdm import tqdm
import json
def create_chunks(documents):
    chunks = []
    for document in tqdm(documents):
        res_string = make_chunks(document)
        try:
            # The LLM returns a string, we must convert it to a Python list
            # We often need to strip markdown ```json code blocks first
            clean_json = res_string.replace("```json", "").replace("```", "").strip()
            res_data = json.loads(clean_json)
            
            if isinstance(res_data, list):
                chunks.extend(res_data)
            else:
                chunks.append(res_data)
        except json.JSONDecodeError as e:
            print(f"Failed to parse document: {e}")
            # Optional: save the 'res_string' to a file to see why it failed
            continue 
    return chunks


In [ ]:
chunks = create_chunks(documents)

In [ ]:
chunks

In [ ]:
with open('output_file.json','w',encoding='utf-8') as f:
    for chunk in chunks:
        f.write(json.dumps(chunk))


In [ ]:
import json
chunk_ai=[]
with open('output_file.json','r',encoding='utf-8') as f:
    chunk_data = f.read()

decoder = json.JSONDecoder()
pos = 0

while pos< len(chunk_data):
    current_remainder = chunk_data[pos:]
    data_to_decode = current_remainder.lstrip()
    pos += len(current_remainder) - len(data_to_decode)
    obj,pos_to_inc = decoder.raw_decode(chunk_data,pos)
    chunk_ai.append(obj)
    pos = pos_to_inc

len(chunk_ai)


In [ ]:
for chunk in chunk_ai[:2]:
    print(','.join(chunk['keywords']))

In [ ]:
from chromadb import PersistentClient
from litellm import completion
from openai import OpenAI

openai = OpenAI(api_key = openai_api_key)
collection_name = 'docs'
def create_embeddings(chunks):
    chroma = PersistentClient(path='vector_store_1')
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)
    
    texts = [chunk['chunk_text'] for chunk in chunk_ai]
    data_to_embed = [f"{' '.join(chunk.get('keywords',[]))} {chunk.get('summary',[])}" for chunk in chunk_ai]
    emb = openai.embeddings.create(model='text-embedding-3-small',input=data_to_embed).data
    vectors = [e.embedding for e in emb]
    ids = [str(i) for i in range(len(chunks))]
    metadata = [{'doc_type':chunk['doc_type']} for chunk in chunk_ai]
    
    collection = chroma.get_or_create_collection('docs')

    collection.add(ids=ids,embeddings=vectors,documents=texts,metadatas=metadata)
    print(f"Vectorstore created with {collection.count()} documents")


In [ ]:
create_embeddings(chunk_ai)

In [ ]:
from pydantic import BaseModel
class Result(BaseModel):
    page_content:str
    metadata:dict

In [ ]:
from chromadb import PersistentClient

chroma = PersistentClient(path='vector_store_1')
collection = chroma.get_or_create_collection('docs')

def fetch_context(question):
    chunks=[]
    query= openai.embeddings.create(model='text-embedding-3-small', input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query],n_results=10)
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks


In [ ]:
question = "Who won the IIOTY award?"
chunks = fetch_context(question)

In [ ]:
chunks

In [ ]:
question = "Who won the IIOTY award?"
RETRIEVAL_K = 20
chunks = fetch_context(question)
for index, c in enumerate(chunks):
    if "award" in c.page_content.lower():
        print(index)

In [ ]:
collection = chroma.get_or_create_collection('docs')
collection.count()

In [ ]:
def calculate_mrr(keyword,reterieved_docs):
    keyword = keyword.lower()
    for rank,doc in enumerate(reterieved_docs):
        if keyword in doc.lower():
            return 1/rank
    return 0

In [ ]:
results['documents'][0]

In [ ]:
from chromadb import PersistentClient

chroma = PersistentClient(path='vector_store_1')
collection = chroma.get_or_create_collection('docs')
question="who is the ceo?"
qemd = openai.embeddings.create(model='text-embedding-3-small',input=[question]).data[0].embedding
results = collection.query(query_embeddings=qemd,n_results=10)
calculate_mrr("Avery",results['documents'][0])

In [ ]:
import math
def calculate_dcg(relevances: list[int], k: int) -> float:
    """Calculate Discounted Cumulative Gain."""
    dcg = 0.0
    for i in range(min(k, len(relevances))):
        dcg += relevances[i] / math.log2(i + 2)  # i+2 because rank starts at 1
    return dcg


def calculate_ndcg(keyword: str, retrieved_docs: list, k: int = 10) -> float:
    """Calculate nDCG for a single keyword (binary relevance, case-insensitive)."""
    keyword_lower = keyword.lower()

    # Binary relevance: 1 if keyword found, 0 otherwise
    relevances = [
        1 if keyword_lower in doc.lower() else 0 for doc in retrieved_docs[:k]
    ]
    print(relevances)

    # DCG
    dcg = calculate_dcg(relevances, k)

    # Ideal DCG (best case: keyword in first position)
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = calculate_dcg(ideal_relevances, k)

    print(idcg)

    return dcg / idcg if idcg > 0 else 0.0

calculate_ndcg("Avery",results['documents'][0])

# Reranker and Query Rewriting

In [ ]:
class RankOrder(BaseModel):
    order:list[int]

In [ ]:
def rerank(question,chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    

    for index,chunk_text in enumerate(chunks):
        user_prompt += f"#Chunk id:{index}: {chunk_text}"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL,messages=messages,response_format =RankOrder )
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i-1] for i in order]


In [ ]:
def fetch_contect(question):
    question_emb = openai.embeddings.create(model="text-embedding-3-small",input=[question]).data[0].embedding
    chroma = PersistentClient(path="vector_store_1")
    collection= chroma.get_or_create_collection("docs")
    results = collection.query(query_embeddings=[question_emb],n_results=10)
    
    chunks = []
    for result in zip(results['documents'][0], results['metadatas'][0]):
        chunks.append(Result(page_content=result[0],metadata=result[1]))
    chunks_m= rerank(question,chunks)
    return chunks_m

c_m = fetch_contect("Who is the CEO?")
c=fetch_context("Who is the CEO?")
c_m

In [ ]:
for i in c_m:
    print(i)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
def make_rag_messsages(question,history,chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['doc_type']}: \n {chunk.page_content}" for chunk in chunks)
    system_prompt=SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("Who won the IIOTY award?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messsages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Who won the IIOTY award?", [])

In [ ]:
answer_question("Who is the ceo? list all of them including co founders", [])